# climate-toolkit — Colab quick start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CGIAR-Climate-Data-Hub/climate-toolkit/blob/main/examples/climate_toolkit_colab.ipynb)

Location-based climate, season, climatology, hazard, and projection analysis, as a Python library.

This notebook accompanies the [Use as a package](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/use-as-a-package/) guide. It runs top-to-bottom on a fresh Colab runtime with **no credentials** — sections 1–4 use NASA POWER and weather stations, which need nothing. Section 5 (optional) unlocks the Earth Engine-backed sources (AgERA5, ERA5, CHIRPS, NEX-GDDP, ...) with a one-time free registration.

**Contents**
1. Install
2. Fetch daily climate data → pandas DataFrame
3. Seasonal climatology & statistics
4. Weather-station observations
5. Optional: Earth Engine sources & hazard assessment
6. Where to go next

## 1. Install

The package is not on PyPI yet, so install straight from GitHub. Takes ~1–2 minutes on Colab.

In [ ]:
%pip install -q "git+https://github.com/CGIAR-Climate-Data-Hub/climate-toolkit.git"

In [ ]:
import climate_toolkit as ct

print(f"climate_toolkit v{ct.__version__}")
print("Public API:", [n for n in ct.__all__ if not n.startswith("__")])

## 2. Fetch daily climate data → pandas DataFrame

`nasa_power` uses plain HTTPS — no credentials, no Earth Engine. We fetch one year of daily rainfall and temperature for Nairobi, Kenya. Swap in your own coordinates.

In [ ]:
from datetime import date

from climate_toolkit.fetch_data.source_data.sources.utils.models import ClimateVariable

LAT, LON = -1.286, 36.817  # Nairobi, Kenya — swap in your own site

df = ct.fetch_climate_data(
    source="nasa_power",
    location_coord=(LAT, LON),
    variables=[
        ClimateVariable.precipitation,
        ClimateVariable.max_temperature,
        ClimateVariable.min_temperature,
    ],
    date_from=date(2020, 1, 1),
    date_to=date(2020, 12, 31),
    verbose=False,
)
print(f"{len(df)} daily rows")
df.head()

In [ ]:
# It's a normal DataFrame — plot, resample, export as usual.
df.set_index("date")["precipitation"].plot(
    figsize=(10, 3), title="Daily precipitation, Nairobi 2020 (NASA POWER)"
);

## 3. Seasonal climatology & statistics

`analyze_climate_statistics` detects growing seasons and returns a nested dict of per-season statistics, water balance (ET0, NDWS, WRSI), and long-term-mean summaries.

> A real climatology needs ~20+ years; the short window here keeps the demo fast and just prints a warning.

In [ ]:
stats = ct.analyze_climate_statistics(
    location_coord=(LAT, LON),
    start_year=2015,
    end_year=2020,
    source="nasa_power",
)
print("Result blocks:", sorted(stats.keys()))

In [ ]:
# Long-term-mean summary per detected season
stats["ltm_season_summary"]

## 4. Weather-station observations

`download_station_data` finds the nearest GHCN-Daily (or GSOD) station and downloads its daily observations. No credentials needed. Arguments are keyword-only.

In [ ]:
station = ct.download_station_data(
    station_source="ghcn_daily",
    station_coord=(LAT, LON),
    date_from=date(2020, 1, 1),
    date_to=date(2020, 12, 31),
    max_distance_km=50.0,
    auto_select="auto-1",
)
station

## 5. Optional: Earth Engine sources & hazard assessment

Most gridded and projection sources (`agera_5`, `era_5`, `chirps_*`, `imerg`, `terraclimate`, `cmip_6`, `nex_gddp`, ...) route through **Google Earth Engine**. One-time setup, **free for noncommercial use** (research, academia, nonprofit):

1. Register a Cloud project at https://code.earthengine.google.com/register — choose **Unpaid usage** + a category like *Academia & Research*. No credit card needed.
2. Set `RUN_EARTH_ENGINE = True` and your project id below, then run the cell. Colab pops up a Google sign-in for `ee.Authenticate()`.

Full walkthrough: [Getting started → Google Earth Engine credentials](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/getting_started/#2-google-earth-engine-credentials).

In [ ]:
RUN_EARTH_ENGINE = False  # set True after registering (free for noncommercial use)
GCP_PROJECT_ID = "your-ee-project-id"  # <-- your registered Cloud project id

if RUN_EARTH_ENGINE:
    import os

    import ee

    ee.Authenticate()  # interactive Google sign-in (works natively in Colab)
    os.environ["GCP_PROJECT_ID"] = GCP_PROJECT_ID
    ee.Initialize(project=GCP_PROJECT_ID)
    print("Earth Engine ready")

In [ ]:
if RUN_EARTH_ENGINE:
    # AgERA5 — recommended default gridded source
    df_ee = ct.fetch_climate_data(
        source="agera_5",
        location_coord=(LAT, LON),
        variables=[
            ClimateVariable.precipitation,
            ClimateVariable.max_temperature,
            ClimateVariable.min_temperature,
        ],
        date_from=date(2020, 1, 1),
        date_to=date(2020, 12, 31),
        verbose=False,
    )
    display(df_ee.head())

    # Crop hazard assessment (heat, drought, waterlogging, ...) for a maize season
    hazards = ct.evaluate_hazards(
        crop_name="Maize",
        location_coord=(LAT, LON),
        date_from="2020-03-01",
        date_to="2020-08-31",
    )
    print("Hazard blocks:", sorted(hazards.keys()))
else:
    print("Skipped — set RUN_EARTH_ENGINE = True in the cell above to enable.")

## 6. Where to go next

- **[Use as a package](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/use-as-a-package/)** — the full guide: all seven public functions, data sources, variables, caching, recipes, troubleshooting.
- **[Getting started](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/getting_started/)** — install and Earth Engine setup in detail.
- **[API reference](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/api/)** — rendered from the docstrings; or run `help(ct.fetch_climate_data)` right here.
- Other entry points not shown above: `compare_climate_periods`, `compare_climate_sources`, `compare_station_to_grids`.

Issues and questions: https://github.com/CGIAR-Climate-Data-Hub/climate-toolkit/issues